# Hands-on solution: next week's log

Every placeholder is filled with the code of the right letter, the notebook has been run from a
clean kernel, and a line under each step says why the other letters fail.

The letters in order: **b d a a c d b c b d c a**.

In [1]:
import pathlib
import sys

here = pathlib.Path.cwd().resolve()
root = next(p for p in [here, *here.parents] if (p / "scripts" / "c2kit.py").exists())
sys.path.insert(0, str(root / "scripts"))
import c2kit as kit

kit.side_by_side(
    kit.ladder(["the librarian's log", "hands-on: next week's log"], lit=1,
               title="Wednesday's notebooks", show=False),
    kit.flow(["find a record", "two running totals", "totals per department",
              "a function with a loan", "read, then fix", "the weekly report"],
             title="Six steps, twelve letters", show=False),
)

The setup cell finds the programme's helper, `scripts/c2kit.py`, by walking up from this folder, and loads the log from `../data/`. This week's log is a different file from the class demo's, so no number carries over.

In [2]:
log = kit.load_records("C2_W00_D03_next_week_STUDENT.py")
print(f"{len(log)} issues loaded")

10 issues loaded


## 1. Find a record, and count the log

A position picks a record and a key picks a field. Lists count from 0.

In [3]:
# TODO 1. Which expression gives the book of the third issue in the log?
#   a) log[3]["book"]
#   b) log[2]["book"]
#   c) log["book"][2]
#   d) log[2]["issue_id"]["book"]
third_book = log[2]["book"]

# TODO 2. Which expression counts the issues in the log?
#   a) len(log[0])
#   b) log.count()
#   c) sum(log)
#   d) len(log)
issues = len(log)
print(third_book, issues)

Algebra 10


In [4]:
kit.check("the third issue is Algebra", third_book == "Algebra", f"got {third_book!r}")
kit.check("the log holds 10 issues", issues == 10, f"got {issues!r}")

**Why:** b holds. a reads the fourth issue, because lists count from 0; c asks the list for a key it does not have; d reads the issue number and then asks that number for a book.

**Why:** d holds. a counts the fields in one record, which is 5; b and c are not how a list reports its length, and neither runs on this log.

## 2. Two running totals in one loop

One loop can carry two totals: a count of the issues kept past the 14-day loan, and a list of the
students seen so far, which grows only the first time a student appears.

In [5]:
kit.flow(["start before the loop", "update once per record", "read after the loop"], lit=1, title="Both totals make the same three moves")

In [6]:
late_count = 0
seen = []
for issue in log:
    if issue["days_kept"] > 14:
        # TODO 3. Which update counts the issues kept past the 14-day loan?
        #   a) late_count = late_count + 1
        #   b) late_count = late_count + issue["days_kept"]
        #   c) late_count = issue["days_kept"] - 14
        #   d) late_count = 1
        late_count = late_count + 1
    if issue["student"] not in seen:
        # TODO 4. Which line records a student the first time they appear?
        #   a) seen.append(issue["student"])
        #   b) seen = issue["student"]
        #   c) seen.append(issue)
        #   d) seen.append(issue["student"] + issue["dept"])
        seen.append(issue["student"])
print(late_count, "late issues;", len(seen), "different students")

4 late issues; 7 different students


In [7]:
kit.check("4 issues were kept past the loan", late_count == 4, f"got {late_count}")
kit.check("7 different students borrowed", len(seen) == 7, f"got {len(seen)}")

**Why:** a holds. b adds the days rather than one issue; c keeps only the last late book's days past the loan; d resets the count to 1 at every late book.

**Why:** a holds. b replaces the whole list with one name; c stores the record, so the name is never found in `seen`; d stores a student and a department glued together, which never matches a name either.

## 3. Totals per department, then days per issue

A dictionary is the running total when the question is "per department". The counts are filled in
for you; the days are yours. Then the average: how long a department's books stay out per issue.

In [8]:
counts = {}
totals = {}
for issue in log:
    dept = issue["dept"]
    if dept not in totals:
        counts[dept] = 0
        # TODO 5. Which line starts a department's running total of days?
        #   a) totals[dept] = issue["days_kept"]
        #   b) totals = {dept: 0}
        #   c) totals[dept] = 0
        #   d) totals[dept] = 1
        totals[dept] = 0
    counts[dept] = counts[dept] + 1
    # TODO 6. Which update adds this issue's days to its department?
    #   a) totals[dept] = totals[dept] + 1
    #   b) totals[dept] = issue["days_kept"]
    #   c) totals[dept] = sum(totals.values()) + issue["days_kept"]
    #   d) totals[dept] = totals[dept] + issue["days_kept"]
    totals[dept] = totals[dept] + issue["days_kept"]

per_issue = {}
for dept in totals:
    # TODO 7. Which expression gives the average days a department's books stay out?
    #   a) totals[dept] / len(log)
    #   b) totals[dept] / counts[dept]
    #   c) counts[dept] / totals[dept]
    #   d) totals[dept] // counts[dept]
    per_issue[dept] = totals[dept] / counts[dept]

kit.table(["department", "issues", "days out", "days per issue"],
          [(d, counts[d], totals[d], round(per_issue[d], 2)) for d in totals],
          caption="Next week's log, by department")

department,issues,days out,days per issue
Science,3,44,14.67
Arts,4,45,11.25
Maths,3,35,11.67


In [9]:
kit.check("the days per department add back to 124", sum(totals.values()) == 124,
          f"got {sum(totals.values())}")
kit.check("Arts keeps books out longest in total, at 45 days", "Arts" in totals and totals["Arts"] == 45)
kit.check("Science keeps them longest per issue, at 14.67 days",
          "Science" in per_issue and round(per_issue["Science"], 2) == 14.67)
kit.check("Arts averages 11.25 days per issue", "Arts" in per_issue and per_issue["Arts"] == 11.25)

**Why:** c holds. a starts at the first issue's days and then adds them again on the next line; b throws away every department already totalled; d starts one day too high.

**Why:** d holds. a counts issues rather than days; b keeps only the last issue's days; c adds every department's days into this one.

**Why:** b holds. a divides by every issue in the log rather than the department's own; c is the right numbers upside down; d drops the decimals, so Science reads 14 and Arts 11.

## 4. A function with a loan period

The librarian may shorten the loan, so the loan becomes a parameter with 14 as its default. Every
book returned on time has to hand back a number too, or the total cannot add it.

In [10]:
kit.flow(["the log goes in", "the loop runs", "return hands the number back"], lit=2, title="A function the library can rerun")

In [11]:
def late_days(issue, loan=14):
    if issue["days_kept"] > loan:
        return issue["days_kept"] - loan
    # TODO 8. What should the function hand back for a book returned on time?
    #   a) return loan
    #   b) print(0)
    #   c) return 0
    #   d) return issue["days_kept"]
    return 0


def total_late(log, loan=14):
    total = 0
    for issue in log:
        # TODO 9. What must the call pass so a shorter loan changes the answer?
        #   a) late_days(issue, 14)
        #   b) late_days(issue, loan)
        #   c) late_days(issue, issue["days_kept"])
        #   d) late_days(issue, total)
        total = total + late_days(issue, loan)
    return total

print(total_late(log), "days late at a 14-day loan;", total_late(log, loan=10), "at a 10-day loan")

16 days late at a 14-day loan; 37 at a 10-day loan


In [12]:
kit.check("16 days late at the 14-day loan", total_late(log) == 16, f"got {total_late(log)}")
kit.check("37 days late if the loan were 10 days", total_late(log, loan=10) == 37,
          f"got {total_late(log, loan=10)}")

**Why:** c holds. a reports every on-time book as 14 days late; b prints and hands back `None`, which breaks the addition exactly as the class demo did; d reports the whole loan as late.

**Why:** b holds. a ignores the loan the caller asked for; c and d pass a number that is not a loan period, so every answer is wrong.

## 5. Read the traceback, then fix the loop

The loop below asks for one position too many. The `try` block catches the error so every line of
its traceback can be printed and read from the bottom up.

In [13]:
kit.stack(["what happened: the last line", "where: the line and its code", "how it got there: the lines above"], lit=0, title="Read it from the bottom up")

In [14]:
import traceback

try:
    for i in range(len(log) + 1):
        last_book = log[i]["book"]
except IndexError:
    message_lines = traceback.format_exc().strip().splitlines()
    for line in message_lines:
        print(line)

Traceback (most recent call last):
  File "/tmp/ipykernel_2638/2631111630.py", line 5, in <module>
    last_book = log[i]["book"]
                ~~~^^^
IndexError: list index out of range


In [15]:
# TODO 10. Which line of the traceback says what happened?
#   a) message_lines[0]
#   b) message_lines[1]
#   c) message_lines[len(message_lines) - 2]
#   d) message_lines[-1]
what_happened = message_lines[-1]

# TODO 11. Which range visits every issue exactly once?
#   a) range(len(log) - 1)
#   b) range(1, len(log) + 1)
#   c) range(len(log))
#   d) range(len(log), 0, -1)
for i in range(len(log)):
    last_book = log[i]["book"]
print(what_happened)
print(last_book)

IndexError: list index out of range
Poetry


In [16]:
kit.check("the last line names the error", what_happened.startswith("IndexError"), what_happened)
kit.check("the loop reached the last issue, Poetry", last_book == "Poetry", f"got {last_book!r}")

**Why:** d holds. a is the opening line every traceback shares; b and c point at where the error happened, which is the second thing to read.

**Why:** c holds. a stops one short and misses Poetry; b and d both ask for position 10 in a list whose last position is 9.

## 6. The weekly report

The librarian's three answers, handed back together so next week's report is one call. This step is
tonight's take-home if the class ends before it.

In [17]:
def weekly_report(log):
    counts = {}
    for issue in log:
        dept = issue["dept"]
        if dept not in counts:
            counts[dept] = 0
        counts[dept] = counts[dept] + 1
    busiest = None
    for dept in counts:
        if busiest is None or counts[dept] > counts[busiest]:
            busiest = dept
    # TODO 12. Which line hands the librarian all three answers at once?
    #   a) return {"issues": len(log), "busiest": busiest, "late_days": total_late(log)}
    #   b) print({"issues": len(log), "busiest": busiest, "late_days": total_late(log)})
    #   c) return [len(log), busiest, total_late(log)]
    #   d) return {"issues": len(log), "busiest": busiest, "late_days": total_late(log, loan=10)}
    return {"issues": len(log), "busiest": busiest, "late_days": total_late(log)}

report = weekly_report(log)
print(report)

{'issues': 10, 'busiest': 'Arts', 'late_days': 16}


In [18]:
kit.check("the report is a dictionary", type(report) is dict, type(report).__name__)
kit.check("it carries the three answers for next week's log",
          report == {"issues": 10, "busiest": "Arts", "late_days": 16}, str(report))

**Why:** a holds. b prints the report and hands back `None`; c returns the answers without the names that say which is which; d uses a 10-day loan the librarian never set.

In [19]:
kit.check_summary()

Every check above passes on the filled notebook. A learner's FAIL line points at the step to revisit.